# ⚙️ Speech-to-Text & Summarization Backend (Strict Manual Inference - Decoder-Only LLM)

This Google Colab notebook implements an automated meeting intelligence backend using **strict manual inference** (NO `pipeline` abstraction).

- **ASR Model:** `openai/whisper-small` instantiated via `AutoProcessor` and `AutoModelForSpeechSeq2Seq`.
- **Summarization Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Decoder-Only Causal LLM) instantiated via `AutoTokenizer` and `AutoModelForCausalLM`.
- **Inference Logic:** Explicit feature extraction, `apply_chat_template` formatting, tokenization, PyTorch tensor manipulation, `model.generate()`, output prompt slicing, and decoding.
- **API Gateway:** FastAPI with ngrok tunnel (identical endpoint signature to the pipeline backend).

---

## 📋 Quickstart Instructions

1. **Enable GPU Acceleration**:
   - In the top menu, select **Runtime** > **Change runtime type**.
   - Set **Hardware accelerator** to **T4 GPU** and click **Save**.

2. **Install Dependencies**:
   - Run **Step 1** below to install all required packages.

3. **Configure Your ngrok Auth Token**:
   - Obtain your free auth token from [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
   - In **Step 5**, paste your token into: `NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"`.

4. **Start the Backend Server**:
   - Run **Step 2**, **Step 3**, **Step 4**, and **Step 5** sequentially.
   - The notebook loads raw models onto the GPU, registers FastAPI endpoints, and opens an ngrok tunnel.

5. **Connect from Local Gradio Client**:
   - Copy the printed public ngrok URL (e.g., `https://xxxx-xx-xx-xx.ngrok-free.app`).
   - Paste it into the `Colab Backend URL` field in your local `client.py` application.
   - Keep this Colab tab active while processing audio files.

## Step 1: Install Required Dependencies

In [ ]:
# Install core machine learning and API server packages
!pip install -q transformers accelerate soundfile librosa fastapi uvicorn pyngrok nest-asyncio python-multipart torch torchaudio

## Step 2: Manually Instantiate Processors, Tokenizers, and Models

Instead of `pipeline()`, we directly instantiate:
- **`AutoProcessor`** & **`AutoModelForSpeechSeq2Seq`** for Whisper.
- **`AutoTokenizer`** & **`AutoModelForCausalLM`** for Qwen2.5 (Decoder-Only architecture).

### Pad Token Handling in Decoder-Only Models:
Many decoder-only models are pre-trained on continuous token streams and do not specify a separate `<pad>` token. When using Hugging Face for batched tensor generation, we ensure `pad_token_id` is assigned (often mapped to `eos_token_id`).

In [ ]:
import torch
from transformers import (
    AutoProcessor,
    AutoModelForSpeechSeq2Seq,
    AutoTokenizer,
    AutoModelForCausalLM,
)

# Verify GPU device availability
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"[*] Compute device: {device} | Torch dtype: {torch_dtype}")
if torch.cuda.is_available():
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")

# 1. Manually Load Whisper ASR Components
whisper_model_id = "openai/whisper-small"
print(f"[*] Loading Whisper Processor and Model ('{whisper_model_id}')...")
asr_processor = AutoProcessor.from_pretrained(whisper_model_id)
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    whisper_model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
).to(device)
asr_model.eval()
print("[✓] Whisper ASR components loaded into GPU memory in eval mode!")

# 2. Manually Load Decoder-Only LLM Components (AutoModelForCausalLM)
llm_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"[*] Loading Decoder-Only Tokenizer and Model ('{llm_model_id}')...")
summarizer_tokenizer = AutoTokenizer.from_pretrained(llm_model_id)

# Handle pad token for decoder-only model if not defined
if summarizer_tokenizer.pad_token_id is None:
    summarizer_tokenizer.pad_token = summarizer_tokenizer.eos_token
    summarizer_tokenizer.pad_token_id = summarizer_tokenizer.eos_token_id

summarizer_model = AutoModelForCausalLM.from_pretrained(
    llm_model_id,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
).to(device)
summarizer_model.eval()
print("[✓] Decoder-Only Causal LM loaded into GPU memory in eval mode!")

## Step 3: Implement Manual Inference Loops

### Essential Mechanics of Decoder-Only Manual Inference:
1. **`apply_chat_template()`**: Formats system and user messages into the model's exact conversation structure.
2. **`add_generation_prompt=True`**: Appends the assistant token header so the model knows it is its turn to respond.
3. **Prompt Slicing (`output_ids[0][prompt_length:]`)**: Because a decoder-only model processes input and output in a single continuous stream, `model.generate()` returns the **entire sequence (prompt + output)**. We must slice off the prompt tokens so the output contains only the generated summary!

In [ ]:
from typing import List, Tuple, Dict
import librosa

def manual_transcribe_audio(audio_path: str) -> Tuple[str, float]:
    """
    Executes manual speech-to-text inference on an audio file without pipeline.
    - Resamples audio to 16kHz mono.
    - Slices audio into 30-second windows (480,000 samples).
    - Extracts log-Mel spectrogram features via AutoProcessor.
    - Invokes model.generate() with torch.no_grad().
    - Decodes token IDs into human-readable text.
    """
    audio_array, sampling_rate = librosa.load(audio_path, sr=16000, mono=True)
    total_samples = len(audio_array)
    audio_duration = float(total_samples) / 16000.0

    if total_samples == 0:
        return "", 0.0

    chunk_size = 16000 * 30  # 30s * 16kHz = 480,000 samples
    transcribed_segments: List[str] = []

    for start_idx in range(0, total_samples, chunk_size):
        end_idx = min(start_idx + chunk_size, total_samples)
        chunk = audio_array[start_idx:end_idx]

        if len(chunk) < 4000:  # Skip fragments < 0.25s
            continue

        # 1. Feature extraction: Convert raw waveform to log-Mel spectrogram
        inputs = asr_processor(
            chunk,
            sampling_rate=16000,
            return_tensors="pt"
        )
        input_features = inputs.input_features.to(device, dtype=torch_dtype)

        # 2. Autoregressive sequence generation
        with torch.no_grad():
            predicted_ids = asr_model.generate(
                input_features,
                max_new_tokens=448,
                num_beams=1,
                do_sample=False
            )

        # 3. Token decoding
        chunk_transcription = asr_processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0].strip()

        if chunk_transcription:
            transcribed_segments.append(chunk_transcription)

    full_transcription = " ".join(transcribed_segments).strip()
    return full_transcription, audio_duration


def manual_summarize_text(text: str, min_length: int = 30, max_length: int = 150) -> str:
    """
    Executes manual summarization using a Decoder-Only Causal LM.
    - Formats prompt with apply_chat_template()
    - Appends generation prompt
    - Generates new tokens autoregressively
    - Slices off the input prompt from the output tensor
    - Decodes only the generated response tokens
    """
    if not text.strip():
        return "No speech detected to summarize."

    messages: List[Dict[str, str]] = [
        {
            "role": "system",
            "content": (
                "You are an expert executive meeting assistant. Provide a concise, clear, and "
                "objective summary of the meeting transcription, highlighting key discussion points, "
                "decisions, and action items. Do not repeat the entire transcript."
            )
        },
        {
            "role": "user",
            "content": f"Please summarize the following meeting transcription:\n\n{text}"
        }
    ]

    # 1. Apply Chat Template to format prompt into token IDs
    # add_generation_prompt=True ensures the assistant role delimiter is appended
    prompt_tokenized = summarizer_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Handle tensor vs dict return types
    if isinstance(prompt_tokenized, torch.Tensor):
        input_ids = prompt_tokenized.to(device)
        attention_mask = torch.ones_like(input_ids).to(device)
    else:
        input_ids = prompt_tokenized["input_ids"].to(device)
        attention_mask = prompt_tokenized["attention_mask"].to(device)

    prompt_length = input_ids.shape[1]

    # 2. Generate new tokens via Causal LM
    with torch.no_grad():
        output_ids = summarizer_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_length,
            min_new_tokens=min(min_length, max_length),
            do_sample=False,
            pad_token_id=summarizer_tokenizer.pad_token_id,
            eos_token_id=summarizer_tokenizer.eos_token_id
        )

    # 3. CRITICAL DECODER-ONLY STEP: Slice off the prompt tokens!
    # Output contains [prompt_tokens + generated_tokens]
    generated_tokens = output_ids[0][prompt_length:]

    # 4. Decode generated tokens into clean text
    decoded_summary = summarizer_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )
    return decoded_summary.strip()

## Step 4: Define FastAPI Application

The API endpoints and response schema match `backend_pipeline.ipynb` identically.

In [ ]:
import os
import time
import tempfile
from typing import Dict, Any
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(
    title="Speech-to-Text & Summarization Backend (Manual Inference - Decoder-Only)",
    description="Production backend using manual PyTorch inference loops with Whisper and Qwen2.5 Decoder-Only LLM",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health_check() -> Dict[str, Any]:
    """Diagnostics endpoint used by the Gradio client."""
    return {
        "status": "ok",
        "mode": "manual",
        "model_asr": "openai/whisper-small",
        "model_summarizer": "Qwen/Qwen2.5-1.5B-Instruct (Decoder-Only)",
        "device": str(device)
    }

@app.post("/process")
async def process_audio(
    file: UploadFile = File(...),
    min_length: int = Form(30),
    max_length: int = Form(150)
) -> Dict[str, Any]:
    """
    Receives an uploaded audio file, executes manual ASR transcription and decoder-only summarization,
    and returns a structured JSON payload with execution telemetry.
    """
    temp_audio_path = None
    try:
        # 1. Save uploaded bytes to disk
        suffix = os.path.splitext(file.filename or "audio.wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as temp_audio:
            content = await file.read()
            temp_audio.write(content)
            temp_audio_path = temp_audio.name

        # 2. Transcribe via manual inference loop
        t_asr_start = time.time()
        transcription, duration = manual_transcribe_audio(temp_audio_path)
        asr_time = round(time.time() - t_asr_start, 2)

        # 3. Summarize via manual decoder-only inference loop
        t_sum_start = time.time()
        if transcription:
            summary = manual_summarize_text(
                transcription,
                min_length=min_length,
                max_length=max_length
            )
        else:
            summary = "No spoken content was detected in the audio file."
        sum_time = round(time.time() - t_sum_start, 2)

        total_time = round(asr_time + sum_time, 2)

        return {
            "status": "success",
            "transcription": transcription,
            "summary": summary,
            "audio_duration_seconds": round(duration, 2),
            "metrics": {
                "transcription_time_seconds": asr_time,
                "summarization_time_seconds": sum_time,
                "total_time_seconds": total_time
            }
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Manual inference processing error: {str(exc)}")
    finally:
        if temp_audio_path and os.path.exists(temp_audio_path):
            try:
                os.remove(temp_audio_path)
            except Exception:
                pass

## Step 5: Configure ngrok Tunnel & Launch Uvicorn Server

Paste your ngrok token below and run the cell. Once the public URL appears, copy and paste it into `client.py`.

In [ ]:
import uvicorn
from pyngrok import ngrok
import nest_asyncio

# 1. Configure ngrok Authentication Token
# Replace with your personal token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

ngrok.kill()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Open Public Tunnel to Port 8000
public_url = ngrok.connect(8000).public_url

print("=" * 65)
print("🚀 Google Colab Manual Backend (Decoder-Only LLM) is ONLINE!")
print(f"[*] Public ngrok URL: {public_url}")
print(f"[*] Health check:     {public_url}/health")
print(f"[*] Process endpoint: {public_url}/process")
print("=" * 65)
print("👉 Copy the Public ngrok URL above and paste it into client.py")
print("=" * 65)

# 3. Run FastAPI ASGI Server
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()